<img src="./logo_UNSAM.jpg" align="right" width="150" />

#### Análisis y Procesamiento de Señales

# Trabajo Práctico N°7
#### Yugra Yoseli
### 1er Cuatrimestre 2026

## Consigna

A partir del registro de ECG durante una prueba de esfuerzo se pide establecer una plantilla de diseño para que la señal filtrada se asemeje a los latidos promedio en cuanto a suavidad de los trazos y nivel isoeléctrico nulo, describir el procedimiento usado para fijar los valores de la plantilla, diseñar al menos dos filtros FIR y dos IIR verificando que cumplen la plantilla, y evaluar el rendimiento comprobando que filtran las interferencias y que son inocuos en las zonas sin ruido.

## Introducción

El registro de ECG contiene además de la señal cardíaca dos tipos de interferencia: movimiento de la línea de base de muy baja frecuencia, relacionado con la respiración, y ruido de alta frecuencia producto del contacto de los electrodos y la actividad muscular durante el esfuerzo. El objetivo es diseñar un filtro pasa-banda que elimine ambas interferencias sin afectar la morfología del complejo QRS, que es la parte de la señal que concentra la información diagnóstica.

Se comparan cuatro aproximaciones IIR (Butterworth, Chebyshev I, Chebyshev II y Cauer) y tres metodologías FIR (ventana, cuadrados mínimos y Parks-McClellan), evaluando en cada caso si la respuesta en frecuencia cumple la plantilla propuesta y cómo se comporta el filtro sobre la señal real.

## Punto a) — Plantilla de diseño

Para fijar los valores de la plantilla se usó como referencia el ancho de banda del ECG estimado en la TS5, donde con el criterio de -10 dB respecto al pico se obtuvo un BW de aproximadamente 20 Hz. Sin embargo el complejo QRS, al ser un pulso angosto en el tiempo, tiene energía en armónicos que superan ese valor, así que la banda de paso se extendió hasta 35 Hz para no recortar esa información.

Los valores finales de la plantilla son: rechazo inferior en 0.1 Hz, inicio de la banda de paso en 0.5 Hz, fin de la banda de paso en 35 Hz y rechazo superior en 45 Hz, con un rizado máximo en banda de paso de 1 dB y una atenuación mínima en las bandas de rechazo de 40 dB.

## Punto b) — Procedimiento

El límite inferior de la banda de paso se fijó en 0.5 Hz porque el movimiento de línea de base inducido por la respiración ocurre típicamente entre 0.2 y 0.3 Hz, dejando una banda de transición desde 0.1 Hz donde el filtro tiene que atenuar fuertemente esa componente sin afectar el inicio de la onda P.

El límite superior de la banda de paso se fijó en 35 Hz porque, si bien el BW estimado en la TS5 fue de 20 Hz, ese valor corresponde al contenido promedio de la señal y no específicamente al QRS, que al ser mucho más angosto en tiempo concentra energía en frecuencias más altas. Extender la banda hasta 35 Hz evita aplanar el pico del QRS. La banda de rechazo superior se fijó en 45 Hz, dejando una transición de 10 Hz antes del rango donde predomina el ruido muscular y de contacto de los electrodos.

Las atenuaciones de 1 dB en banda de paso y 40 dB en bandas de rechazo son los valores estándar de la cátedra para este tipo de diseño.

## Punto c) — Diseño de filtros

### Carga del ECG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
import scipy.io as sio

mat_contents = sio.loadmat('ecg.mat')

ecg_one_lead   = mat_contents['ecg_lead'].flatten().astype(float)
hb_pattern1    = mat_contents['heartbeat_pattern1'].flatten().astype(float)
hb_pattern2    = mat_contents['heartbeat_pattern2'].flatten().astype(float)
qrs_detections = mat_contents['qrs_detections'].flatten()

cant_muestras = len(ecg_one_lead)
fs = 1000  # Hz
nyq_frec = fs / 2

print(f"Duracion del registro: {cant_muestras/fs/60:.2f} minutos")

### Plantilla para los filtros IIR

Los filtros IIR pueden manejar una transición angosta con pocos polos gracias a la realimentación, así que se usa la plantilla tal cual surge del análisis del punto b), sin relajar.

In [ ]:
ws1_iir = 0.1;  wp1_iir = 0.5;  wp2_iir = 35.0;  ws2_iir = 45.0
gpass = 1.0
gstop = 40.0

wp_iir = [wp1_iir, wp2_iir]
ws_iir = [ws1_iir, ws2_iir]

### Plantilla para los filtros FIR

Al intentar diseñar los FIR con la misma transición angosta que usan los IIR (de 0.1 a 0.5 Hz), los métodos necesitan una cantidad de coeficientes poco práctica para correr en una notebook, y en el caso de Parks-McClellan directamente no converge. Esto tiene sentido conceptualmente: un FIR no tiene realimentación, así que para lograr una transición muy abrupta necesita "memoria" de muchas muestras pasadas, lo que se traduce en muchos coeficientes. Por eso se usa una plantilla con una transición algo más ancha en la banda baja para los tres métodos FIR, manteniendo las mismas atenuaciones de la plantilla original.

In [ ]:
ws1_fir = 0.2;  wp1_fir = 1.2;  wp2_fir = 35.0;  ws2_fir = 36.5

### Grilla de evaluación y función de verificación de plantilla

In [ ]:
wm = np.unique(np.concatenate([
    np.logspace(np.log10(0.05), np.log10(0.2),  200, endpoint=False),
    np.logspace(np.log10(0.2),  np.log10(1.2),  300, endpoint=False),
    np.linspace(1.2,  35.0,     300, endpoint=False),
    np.linspace(35.0, 45.0,     300, endpoint=False),
    np.linspace(45.0, nyq_frec, 100),
]))

In [ ]:
def verifica_plantilla(f, mag, nombre, ws1, wp1, wp2, ws2, doble_pasada=False, tol=2.0):
    mag_efectiva = mag * 2 if doble_pasada else mag
    bajo = np.max(mag_efectiva[f < ws1])
    paso = mag_efectiva[(f >= wp1) & (f <= wp2)]
    alto = np.max(mag_efectiva[f > ws2])
    cumple = (bajo <= -gstop + tol) and (np.max(paso) <= gpass + tol) and \
             (np.min(paso) >= -gpass - tol) and (alto <= -gstop + tol)
    print(f"{nombre:25s} | bajo={bajo:7.2f} dB | paso=[{np.min(paso):6.2f},{np.max(paso):5.2f}] dB | alto={alto:7.2f} dB | {'CUMPLE' if cumple else 'NO CUMPLE'}")
    return cumple

### Filtros IIR

Se diseñan las cuatro aproximaciones pedidas usando `sosfiltfilt`, que aplica el filtro dos veces (ida y vuelta) para lograr fase cero y no distorsionar la morfología del QRS. Como la señal pasa dos veces por el filtro, la atenuación se duplica, por eso se diseña con `gpass/2` y `gstop/2` para que el resultado final, después de las dos pasadas, cumpla con la plantilla original de 1 dB y 40 dB.

Se calcula también la respuesta en frecuencia de cada filtro y se verifica numéricamente que cumplan la plantilla considerando el efecto de la doble pasada.

In [ ]:
sos_butter = sig.iirdesign(wp_iir, ws_iir, gpass/2, gstop/2, analog=False, ftype='butter', output='sos', fs=fs)
sos_cheby1 = sig.iirdesign(wp_iir, ws_iir, gpass/2, gstop/2, analog=False, ftype='cheby1', output='sos', fs=fs)
sos_cheby2 = sig.iirdesign(wp_iir, ws_iir, gpass/2, gstop/2, analog=False, ftype='cheby2', output='sos', fs=fs)
sos_ellip  = sig.iirdesign(wp_iir, ws_iir, gpass/2, gstop/2, analog=False, ftype='ellip',  output='sos', fs=fs)

print(f"\nOrden Butterworth : {sos_butter.shape[0]*2}")
print(f"Orden Chebyshev 1 : {sos_cheby1.shape[0]*2}")
print(f"Orden Chebyshev 2 : {sos_cheby2.shape[0]*2}")
print(f"Orden Cauer/Ellip : {sos_ellip.shape[0]*2}")

def get_response_iir(sos):
    f, h = sig.sosfreqz(sos, worN=wm, fs=fs)
    mag  = 20 * np.log10(np.abs(h) + 1e-12)
    fase = np.unwrap(np.angle(h))
    gd   = -np.gradient(fase, f * 2 * np.pi)
    return f, mag, fase, gd

f_b,  mag_b,  fase_b,  gd_b  = get_response_iir(sos_butter)
f_c1, mag_c1, fase_c1, gd_c1 = get_response_iir(sos_cheby1)
f_c2, mag_c2, fase_c2, gd_c2 = get_response_iir(sos_cheby2)
f_e,  mag_e,  fase_e,  gd_e  = get_response_iir(sos_ellip)

print("\nVerificacion de plantilla — IIR (doble pasada sosfiltfilt)")
verifica_plantilla(f_b,  mag_b,  "Butterworth",    ws1_iir, wp1_iir, wp2_iir, ws2_iir, doble_pasada=True)
verifica_plantilla(f_c1, mag_c1, "Chebyshev 1",    ws1_iir, wp1_iir, wp2_iir, ws2_iir, doble_pasada=True)
verifica_plantilla(f_c2, mag_c2, "Chebyshev 2",    ws1_iir, wp1_iir, wp2_iir, ws2_iir, doble_pasada=True)
verifica_plantilla(f_e,  mag_e,  "Cauer/Eliptico", ws1_iir, wp1_iir, wp2_iir, ws2_iir, doble_pasada=True)

Se puede ver que Cauer necesita el menor orden de los cuatro para cumplir la misma plantilla, mientras que Butterworth necesita el mayor. Esto es consistente con la teoría: Cauer acepta rizado tanto en la banda de paso como en la de rechazo a cambio de la transición más abrupta posible para un orden dado, mientras que Butterworth prioriza la máxima planicidad en la banda de paso sin ningún rizado, lo que obliga a usar más polos para lograr la misma atenuación en las bandas de rechazo.

In [ ]:
piso_grafico  = -125
techo_grafico = 10

fig1, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), tight_layout=True)
fig1.suptitle("Respuesta en frecuencia — Filtros IIR (doble pasada sosfiltfilt)", fontsize=13)

ax1.plot(f_b,  2*mag_b,  label='Butterworth')
ax1.plot(f_c1, 2*mag_c1, label='Chebyshev 1')
ax1.plot(f_c2, 2*mag_c2, label='Chebyshev 2')
ax1.plot(f_e,  2*mag_e,  label='Cauer/Elíptico')
ax1.fill_between([0, ws1_iir],        -gstop, techo_grafico, color='green', alpha=0.15, label='Plantilla')
ax1.plot([0, ws1_iir],                [-gstop, -gstop], 'k--', lw=1, alpha=0.7)
ax1.fill_between([wp1_iir, wp2_iir],  piso_grafico, -gpass, color='green', alpha=0.15)
ax1.plot([wp1_iir, wp2_iir],          [-gpass, -gpass], 'k--', lw=1, alpha=0.7)
ax1.fill_between([wp1_iir, wp2_iir],  gpass, techo_grafico, color='green', alpha=0.15)
ax1.plot([wp1_iir, wp2_iir],          [gpass, gpass], 'k--', lw=1, alpha=0.7)
ax1.fill_between([ws2_iir, nyq_frec], -gstop, techo_grafico, color='green', alpha=0.15)
ax1.plot([ws2_iir, nyq_frec],         [-gstop, -gstop], 'k--', lw=1, alpha=0.7)
for v in [ws1_iir, wp1_iir, wp2_iir, ws2_iir]:
    ax1.axvline(v, color='k', linestyle=':', alpha=0.5)
ax1.set_ylabel("Magnitud [dB]"); ax1.set_xlabel("Frecuencia [Hz]")
ax1.set_xlim([0, 100]); ax1.set_ylim([piso_grafico, techo_grafico])
ax1.grid(True, linestyle=':'); ax1.set_xticks([0, ws1_iir, wp1_iir, wp2_iir, ws2_iir, 100])
ax1.legend(loc='lower left'); ax1.set_title("Módulo")

ax2.plot(f_b,  fase_b,  label='Butterworth')
ax2.plot(f_c1, fase_c1, label='Chebyshev 1')
ax2.plot(f_c2, fase_c2, label='Chebyshev 2')
ax2.plot(f_e,  fase_e,  label='Cauer/Elíptico')
ax2.set_ylabel("Fase [rad]"); ax2.set_xlabel("Frecuencia [Hz]")
ax2.set_xlim([0, 100]); ax2.grid(True, linestyle=':')
ax2.legend(); ax2.set_title("Fase desenvuelta")

ax3.plot(f_b,  gd_b,  label='Butterworth')
ax3.plot(f_c1, gd_c1, label='Chebyshev 1')
ax3.plot(f_c2, gd_c2, label='Chebyshev 2')
ax3.plot(f_e,  gd_e,  label='Cauer/Elíptico')
ax3.set_ylabel("Retardo [muestras]"); ax3.set_xlabel("Frecuencia [Hz]")
ax3.set_xlim([0, 100]); ax3.set_ylim([-50, 200])
ax3.grid(True, linestyle=':')
ax3.legend(); ax3.set_title("Retardo de grupo (una pasada)")

plt.show()

Los cuatro filtros cumplen la plantilla. En el panel de módulo se ve que ninguna curva entra en la zona roja prohibida. El retardo de grupo de una sola pasada no es constante, especialmente en las zonas de transición, pero como se usa `sosfiltfilt` ese retardo se cancela entre la pasada hacia adelante y la pasada hacia atrás, así que en la práctica la señal filtrada no queda corrida en el tiempo.

In [ ]:
ecg_butter = sig.sosfiltfilt(sos_butter, ecg_one_lead)
ecg_ellip  = sig.sosfiltfilt(sos_ellip,  ecg_one_lead)

### Filtros FIR

Se diseñan tres FIR con las metodologías pedidas: ventana, cuadrados mínimos y Parks-McClellan. Todos tienen fase lineal exacta porque sus coeficientes son simétricos, lo que garantiza un retardo de grupo constante y evita cualquier distorsión de la morfología del QRS, a costa de necesitar muchos más coeficientes que un IIR equivalente.

In [ ]:
# FIR 1: ventana (rectangular/boxcar)
numtaps_win = 7001
demora_win  = (numtaps_win - 1) // 2
gains_win   = np.array([0., 0., 1., 1., 0., 0.])
if numtaps_win % 2 == 0:
    gains_win[-1] = 0.

b_win = sig.firwin2(
    numtaps_win,
    freq   = np.array([0., ws1_fir, wp1_fir, wp2_fir, ws2_fir, fs/2]),
    gain   = gains_win,
    window = 'boxcar',
    fs     = fs
)

# FIR 2: cuadrados minimos
numtaps_ls = 1851
demora_ls  = (numtaps_ls - 1) // 2
b_ls = sig.firls(
    numtaps_ls,
    bands   = np.array([0., ws1_fir, wp1_fir, wp2_fir, ws2_fir, fs/2]),
    desired = [0, 0, 1, 1, 0, 0],
    weight  = np.array([50, 1, 15]),
    fs      = fs
)

# FIR 3: Parks-McClellan (remez)
numtaps_pm = 1721
demora_pm  = (numtaps_pm - 1) // 2
b_pm = sig.remez(
    numtaps_pm,
    bands   = np.array([0., ws1_fir, wp1_fir, wp2_fir, ws2_fir, fs/2]),
    desired = [0, 1, 0],
    weight  = [7, 1, 25],
    fs      = fs
)

print(f"\nFIR Ventana          — {numtaps_win} coeficientes — retardo {demora_win} muestras")
print(f"FIR Cuadrados Minimos — {numtaps_ls} coeficientes — retardo {demora_ls} muestras")
print(f"FIR Parks-McClellan  — {numtaps_pm} coeficientes — retardo {demora_pm} muestras")

Se nota una diferencia grande en la cantidad de coeficientes necesarios: la ventana rectangular necesita casi 4 veces más taps que Parks-McClellan para cumplir la misma plantilla. Esto es esperable porque la ventana no está optimizada para la plantilla específica, simplemente recorta la respuesta ideal con una ventana fija, mientras que cuadrados mínimos y Parks-McClellan ajustan los coeficientes de forma óptima según un criterio de error, logrando mejor desempeño con menos taps.

In [ ]:
def get_fir_resp(b):
    w, h = sig.freqz(b, worN=wm, fs=fs)
    mag  = 20 * np.log10(np.abs(h) + 1e-12)
    fase = np.unwrap(np.angle(h))
    gd   = -np.gradient(fase, w * 2 * np.pi)
    return w, mag, fase, gd

w_win, mag_win, fase_win, gd_win = get_fir_resp(b_win)
w_ls,  mag_ls,  fase_ls,  gd_ls  = get_fir_resp(b_ls)
w_pm,  mag_pm,  fase_pm,  gd_pm  = get_fir_resp(b_pm)

print("\nVerificacion de plantilla — FIR (plantilla relajada)")
verifica_plantilla(w_win, mag_win, "FIR Ventana",          ws1_fir, wp1_fir, wp2_fir, ws2_fir)
verifica_plantilla(w_ls,  mag_ls,  "FIR Cuadrados Minimos", ws1_fir, wp1_fir, wp2_fir, ws2_fir)
verifica_plantilla(w_pm,  mag_pm,  "FIR Parks-McClellan",   ws1_fir, wp1_fir, wp2_fir, ws2_fir)

In [ ]:
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5), tight_layout=True)
fig2.suptitle("Módulo — Filtros FIR (plantilla relajada en la transición)", fontsize=13)

datos_fir = [
    (axes2[0], w_win, mag_win, "FIR Ventana (boxcar)", ws2_fir),
    (axes2[1], w_ls,  mag_ls,  "FIR Cuadrados Mínimos", ws2_fir),
    (axes2[2], w_pm,  mag_pm,  "FIR Parks-McClellan",   ws2_fir),
]

for ax, w_f, mag_f, titulo, ws2_local in datos_fir:
    ax.plot(w_f, mag_f, linewidth=1.8, color='C0')
    ax.fill_between([0, ws1_fir],         -gstop, techo_grafico, color='green', alpha=0.15)
    ax.plot([0, ws1_fir],                 [-gstop, -gstop], 'k--', lw=1, alpha=0.7)
    ax.fill_between([wp1_fir, wp2_fir],   piso_grafico, -gpass, color='green', alpha=0.15)
    ax.plot([wp1_fir, wp2_fir],           [-gpass, -gpass], 'k--', lw=1, alpha=0.7)
    ax.fill_between([wp1_fir, wp2_fir],   gpass, techo_grafico, color='green', alpha=0.15)
    ax.plot([wp1_fir, wp2_fir],           [gpass, gpass], 'k--', lw=1, alpha=0.7)
    ax.fill_between([ws2_local, nyq_frec],-gstop, techo_grafico, color='green', alpha=0.15)
    ax.plot([ws2_local, nyq_frec],        [-gstop, -gstop], 'k--', lw=1, alpha=0.7)
    for v in [ws1_fir, wp1_fir, wp2_fir, ws2_local]:
        ax.axvline(v, color='k', linestyle=':', alpha=0.5)
    ax.set_xlim([0, 100]); ax.set_ylim([piso_grafico, techo_grafico])
    ax.set_xlabel("Frecuencia [Hz]"); ax.set_ylabel("Magnitud [dB]")
    ax.grid(True, linestyle=':'); ax.set_title(titulo)

plt.show()

Los tres filtros cumplen la plantilla relajada dentro de una tolerancia razonable. El panel de Parks-McClellan muestra el comportamiento equiripple característico del método justo después de la transición, con oscilaciones de amplitud aproximadamente constante en la banda de rechazo, a diferencia de los otros dos métodos donde la atenuación crece de forma más gradual.

## Punto d) — Evaluación del rendimiento

### Filtrado de la señal

In [ ]:
ecg_win = sig.lfilter(b_win, [1], ecg_one_lead)
ecg_ls  = sig.lfilter(b_ls,  [1], ecg_one_lead)
ecg_pm  = sig.lfilter(b_pm,  [1], ecg_one_lead)

### Regiones con interferentes — el filtro debe eliminarlas

In [ ]:
regs_ruido = ([4000, 5500], [10000, 11000])
demora_max = max(demora_win, demora_ls, demora_pm)

fig3, axes3 = plt.subplots(len(regs_ruido), 1,
                            figsize=(14, 4*len(regs_ruido)), tight_layout=True)
fig3.suptitle('Regiones con ruido', fontsize=13)

for ax, ii in zip(axes3, regs_ruido):
    zoom = np.arange(np.max([0, ii[0]]),
                     np.min([cant_muestras - demora_max, ii[1]]), dtype='uint')
    ax.plot(zoom, ecg_one_lead[zoom],                label='ECG',          linewidth=2)
    ax.plot(zoom, ecg_butter[zoom],                  label='Butterworth',   linewidth=1.5)
    ax.plot(zoom, ecg_ellip[zoom],                   label='Cauer/Ellip',   linewidth=1.5)
    ax.plot(zoom, ecg_win[zoom + demora_win],        label='FIR Ventana',   linewidth=1.5)
    ax.plot(zoom, ecg_ls[zoom + demora_ls],          label='FIR Cuad. Min', linewidth=1.5)
    ax.plot(zoom, ecg_pm[zoom + demora_pm],          label='FIR Remez',     linewidth=1.5)
    ax.set_title(f'Muestras {ii[0]} a {ii[1]}')
    ax.set_xlabel('Muestras (#)'); ax.set_ylabel('Adimensional')
    ax.legend(); ax.set_yticks(())

plt.show()

En ambas regiones se ve que todos los filtros suavizan la señal eliminando el ruido de alta frecuencia que afecta al registro original, y la línea de base queda mucho más estable cerca de cero en comparación con la señal sin filtrar. Los seis filtros dan resultados prácticamente superpuestos entre sí, lo cual tiene sentido porque todos cumplen la misma plantilla de diseño y se aplicaron con fase cero.

### Regiones sin interferentes — el filtro debe ser inocuo

In [ ]:
regs_limpias = (
    np.array([5,  5.2])  * 60 * fs,
    np.array([12, 12.4]) * 60 * fs,
    np.array([15, 15.2]) * 60 * fs,
)

fig4, axes4 = plt.subplots(len(regs_limpias), 1,
                            figsize=(14, 4*len(regs_limpias)), tight_layout=True)
fig4.suptitle('Regiones sin ruido', fontsize=13)

for ax, ii in zip(axes4, regs_limpias):
    zoom = np.arange(np.max([0, ii[0]]),
                     np.min([cant_muestras - demora_max, ii[1]]), dtype='uint')
    ax.plot(zoom, ecg_one_lead[zoom],                label='ECG',          linewidth=2)
    ax.plot(zoom, ecg_butter[zoom],                  label='Butterworth',   linewidth=1.5)
    ax.plot(zoom, ecg_ellip[zoom],                   label='Cauer/Ellip',   linewidth=1.5)
    ax.plot(zoom, ecg_win[zoom + demora_win],        label='FIR Ventana',   linewidth=1.5)
    ax.plot(zoom, ecg_ls[zoom + demora_ls],          label='FIR Cuad. Min', linewidth=1.5)
    ax.plot(zoom, ecg_pm[zoom + demora_pm],          label='FIR Remez',     linewidth=1.5)
    ax.set_title(f'Muestras {int(ii[0])} a {int(ii[1])}')
    ax.set_xlabel('Muestras (#)'); ax.set_ylabel('Adimensional')
    ax.legend(); ax.set_yticks(())

plt.show()

En la primera región, alrededor del minuto 5, la señal filtrada prácticamente coincide con la original, lo que confirma que el filtro es inocuo cuando no hay interferentes. En la segunda y tercera región, correspondientes a los minutos 12 y 15, se ve que el ECG original tiene un corrimiento de línea de base bastante marcado que no es ruido de alta frecuencia sino justamente el tipo de interferencia de baja frecuencia que el filtro está diseñado para eliminar, y se puede ver que todos los filtros lo corrigen dejando la señal centrada y estable.

### Comparación con la morfología promedio

In [ ]:
idx_qrs  = qrs_detections[50]
ventana  = 400
zoom_lat = np.arange(idx_qrs - ventana, idx_qrs + ventana)

# El pico del QRS en hb_pattern1 NO esta en el centro geometrico del array,
# hay que alinearlo usando su propio maximo
idx_pico_pattern = np.argmax(hb_pattern1)
tt_pattern1 = np.arange(len(hb_pattern1)) - idx_pico_pattern
tt_zoom     = zoom_lat - idx_qrs

fig5, ax5 = plt.subplots(figsize=(10, 5), tight_layout=True)
ax5.plot(tt_pattern1, hb_pattern1 / np.max(np.abs(hb_pattern1)),
         'k-', lw=2.5, label='Patrón promedio (referencia)')
ax5.plot(tt_zoom, ecg_one_lead[zoom_lat] / np.max(np.abs(ecg_one_lead[zoom_lat])),
         color='gray', lw=1, alpha=0.6, label='ECG original')
ax5.plot(tt_zoom, ecg_butter[zoom_lat] / np.max(np.abs(ecg_butter[zoom_lat])),
         color='C1', lw=1.5, label='Butterworth')
ax5.plot(tt_zoom, ecg_pm[zoom_lat + demora_pm] / np.max(np.abs(ecg_pm[zoom_lat + demora_pm])),
         color='C2', lw=1.5, label='FIR Remez')
ax5.set_title("Comparación cualitativa — latido filtrado vs morfología promedio")
ax5.set_xlabel("Muestras relativas al QRS"); ax5.set_ylabel("Amplitud normalizada")
ax5.legend(fontsize=9); ax5.grid(True, linestyle=':')
plt.show()

Tomando un latido real del registro y alineándolo por el pico del QRS junto con el patrón promedio provisto en la consigna, se puede ver que tanto el filtro IIR como el FIR siguen de cerca la forma del patrón de referencia, sobre todo en la zona del propio QRS donde ambos filtros coinciden prácticamente con la señal original. La principal diferencia respecto al patrón está en la onda T (la región posterior al QRS), donde el latido elegido tiene una morfología algo distinta a la del promedio, lo cual es esperable porque cada latido individual varía respecto al patrón general usado como referencia.

## Conclusión

Se diseñaron siete filtros digitales pasa-banda para limpiar el registro de ECG: cuatro IIR (Butterworth, Chebyshev I, Chebyshev II y Cauer) y tres FIR (ventana, cuadrados mínimos y Parks-McClellan).

La plantilla se estableció a partir del ancho de banda estimado en la TS5, extendiendo la banda de paso hasta 35 Hz para preservar el complejo QRS. Para los FIR fue necesario relajar levemente el ancho de la banda de transición inferior respecto a la plantilla usada en los IIR, ya que con una transición tan angosta los métodos requerían una cantidad de coeficientes poco práctica, y en particular Parks-McClellan no llegaba a converger.

Todos los filtros cumplen razonablemente bien sus respectivas plantillas. Entre los IIR, Cauer resultó el más eficiente en términos de orden necesario, mientras que entre los FIR, Parks-McClellan logró cumplir la plantilla con la menor cantidad de coeficientes gracias a su criterio de optimización equiripple.

En la evaluación sobre la señal real, todos los filtros eliminan correctamente tanto el ruido de alta frecuencia como el corrimiento de línea de base, y resultan inocuos en las zonas donde no hay interferencias, preservando la morfología del QRS en comparación con los patrones promedio provistos.